# Interval-anchored generation metrics

This notebook loads the committed F1@any and F1@max results. Prediction-anchored accuracy is precision-like; GT-anchored accuracy is recall-like. Their harmonic mean is computed per video and macro-averaged over GT-positive videos. See `README.md` for the exact matching rules and limitations.

The committed JSON includes per-video details and SHA-256 hashes of every raw source artifact. To recompute from the preserved legacy raw outputs, run `PYTHONPATH=src .venv/bin/python scripts/evaluate_overlap_metrics.py` from the repository root.

In [ ]:
import json
from pathlib import Path

import pandas as pd

here = Path.cwd()
archive_dir = here if (here / 'results' / 'current').exists() else here / 'notebooks' / 'archive'
result_dir = archive_dir / 'results' / 'current'
summary = pd.read_csv(result_dir / 'overlap_metrics_summary.csv')
selected = pd.read_csv(result_dir / 'overlap_metrics_selected_components.csv')
components = pd.read_csv(result_dir / 'overlap_metrics_by_actor_granularity.csv')
breakdown = pd.read_csv(result_dir / 'overlap_metrics_f1_breakdown.csv')
details = json.loads((result_dir / 'overlap_metric_results.json').read_text())
iou_details = json.loads((result_dir / 'v3_1_spec_eval_results.json').read_text())

## Current v3.1 across IoU thresholds and lenient overlap modes

IoU F1 uses greedy one-to-one segment matching. `any` and `max` judge anchors independently and are not additional IoU thresholds.

In [ ]:
iou = pd.DataFrame(iou_details['structured_prediction_cvs_context'])[['actor', 'granularity', 'iou', 'f1_mean']]
iou = iou[iou.actor.isin(['left', 'right', 'camera'])].pivot(index=['actor', 'granularity'], columns='iou', values='f1_mean')
iou = iou.rename(columns={0.1: 'f1_iou_0.10', 0.25: 'f1_iou_0.25', 0.5: 'f1_iou_0.50'})
overlap = components[(components.pipeline == 'v3.1') & components.actor.isin(['left', 'right', 'camera'])]
overlap = overlap.pivot(index=['actor', 'granularity'], columns='overlap_mode', values='f1_mean').rename(columns={'any': 'f1_any', 'max': 'f1_max'})
threshold_comparison = iou.join(overlap).reset_index()
threshold_comparison['level'] = threshold_comparison.granularity.replace({'exact': 'raw'})
display(threshold_comparison[['actor', 'level', 'f1_iou_0.10', 'f1_iou_0.25', 'f1_iou_0.50', 'f1_any', 'f1_max']].round(3))

In [ ]:
display(summary.round(3))
display(selected.round(3))
display(breakdown.round(3))

In [ ]:
# Current v3.1 release-score components, including both anchor directions.
display(components[(components.pipeline == 'v3.1') & components.apply(lambda row: (row.actor, row.granularity) in {('left', 'exact'), ('right', 'medium'), ('camera', 'coarse')}, axis=1)].round(3))

In [ ]:
# Exact metric policy and raw-artifact paths/hashes used for recomputation.
details['metric_policy'], pd.DataFrame(details['provenance']).T